In [75]:
import numpy as np
from scipy.optimize import linprog

In [76]:
buses = ["A", "B", "C", "D"]

lines = [
    ("A", "B"),
    ("B", "C"),
    ("C", "D"),
    ("A", "D"),
    ("B", "D"),
]

K = np.matrix(np.zeros( (len(lines),len(buses)) ))

for i, (start, end) in enumerate(lines):    
    K[i, buses.index(start)] = 1
    K[i, buses.index(end)] = -1
    #print(f"{index}, {start}, {end}")


B = K.T @ K
B_r = B[:3, :3]
theta = np.linalg.solve(B_r, np.eye(3))
theta_full = np.vstack( (theta, np.zeros(3)) )
ptdf = K @ theta_full

In [78]:
# Part 2 — Contingency analysis
# 5. Calculate the LODFs associated with the B–D outage.
# 6. Use those LODFs to construct the post-outage PTDF matrix.
# 7. Independently rebuild the network without B–D and verify that the two post-outage PTDF matrices agree using np.allclose()

lodf = np.array(ptdf[:4,1]/(1-ptdf[4,1]))

po_ptdf = ptdf[:4, :4] + np.outer(lodf, ptdf[4,:])

po_ptdf


matrix([[ 0.25, -0.5 , -0.25],
        [ 0.25,  0.5 , -0.25],
        [ 0.25,  0.5 ,  0.75],
        [ 0.75,  0.5 ,  0.25]])

In [97]:
# manual post-outage ptdf/ check

lines_ = [
    ("A", "B"),
    ("B", "C"),
    ("C", "D"),
    ("A", "D")
    #("B", "D"),
]

K_ = np.matrix(np.zeros( (len(lines_),len(buses)) ))

for i, (start, end) in enumerate(lines_):    
    K_[i, buses.index(start)] = 1
    K_[i, buses.index(end)] = -1
    #print(f"{index}, {start}, {end}")


B_ = K_.T @ K_
B_r2 = B_[:3, :3]
theta_ = np.linalg.solve(B_r2, np.eye(3))
theta_full_ = np.vstack( (theta_, np.zeros(3)) )
ptdf_ = K_ @ theta_full_

np.allclose(ptdf_, po_ptdf)

True

In [ ]:
# Part 3 — Economic dispatch with SciPy
# 8. Find the least-cost dispatch without the N−1 security requirement. 
# Calculate total generation cost and determine whether the A–D line would exceed its limit following the B–D outage.

c = [20, 65, 120]
A_ub = [[.625, .25, .125]]
b_ub = 10.5
A_eq = [[1, 1, 1]]
b_eq = [16]
bounds = [(0, 16), (0, 16), (0, 16)]

result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

print("Success:", result.success)
print("Generation at A and C:", result.x)
print("Total cost:", result.fun)

# this exceeds the AD line limit if BD trips (.75 * 16 = 12)

Success: True
Generation at A and C: [16.  0.  0.]
Total cost: 320.0


In [124]:

# 9. Set up and solve a security-constrained economic dispatch using scipy.optimize.linprog. 
# Your optimization must include generation costs, energy balance, generator bounds, normal transmission constraints, and the B–D outage constraints.
# 10. Report the optimal N−1 secure dispatch, total system cost, and additional cost of maintaining N−1 security.

c = [20, 65, 120]
A_ub = [[.625, .25, .125], [.75, .5, .25]]

b_ub = [10.5, 10.5]
A_eq = [[1, 1, 1]]
b_eq = [16]
bounds = [(0, 16), (0, 16), (0, 16)]

result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

print("Success:", result.success)
print("Generation at A and C:", result.x)
print("Total cost:", result.fun)

# $320 versus $590 - cost of security is 270
# N-1 dispatch: 10 on gen a and 6 on B

print(result.ineqlin.residual)
print(result.ineqlin.marginals)
print(result.eqlin.marginals)

Success: True
Generation at A and C: [10.  6.  0.]
Total cost: 590.0
[2.75 0.  ]
[  -0. -180.]
[155.]


In [ ]:
# increase AD capacity to 11 and re-optimize
b_ub = [11, 11]

result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

print("Success:", result.success)
print("Generation at A and C:", result.x)
print("Total cost:", result.fun)



print(result.ineqlin.residual)
print(result.ineqlin.marginals)
print(result.eqlin.marginals)

# restore to 10.5

b_ub = [10.5, 10.5]


result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
    method="highs"
)

print("Success:", result.success)
print("Generation at A and C:", result.x)
print("Total cost:", result.fun)

print(result.ineqlin.residual)
print(result.ineqlin.marginals)
print(result.eqlin.marginals)

# LMP at D is 155

mu = -180 # shadow price
155 + .625 * mu # LMP at A = 42.5
155 + .25 * mu # LMP at B = 110
155 + .125 * mu # LMP at C = 132.5

Success: True
Generation at A and C: [12.  4.  0.]
Total cost: 500.0
[2.5 0. ]
[  -0. -180.]
[155.]
Success: True
Generation at A and C: [10.  6.  0.]
Total cost: 590.0
[2.75 0.  ]
[  -0. -180.]
[155.]


132.5